# Chapter 19: Training and Deploying TensorFlow Models at Scale
## Part 1: Serving TensorFlow Models (First 25%)

This notebook covers the fundamental concepts of deploying TensorFlow models to production, specifically:
- **Serving models with TensorFlow Serving**
- **Exporting models to SavedModel format**
- **Inspecting SavedModels**
- **Querying models via REST and gRPC APIs**

### What is TensorFlow Serving?

**TensorFlow Serving** is a production-ready, battle-tested C++ model server designed for serving machine learning models at scale. It provides several critical features:

- ✅ **High Performance**: Sustain high load with low latency
- ✅ **Version Management**: Serve multiple model versions simultaneously
- ✅ **Auto-Deployment**: Automatically detect and deploy latest model versions
- ✅ **Graceful Transitions**: Handle model updates without dropping requests
- ✅ **REST & gRPC APIs**: Flexible query interfaces

Let's explore these concepts step by step with working code examples!

## Section 1: Setup and Installation

First, we need to install and import all necessary libraries. We'll need:
- **TensorFlow**: For model creation and training
- **requests**: For REST API communication
- **tensorflow-serving-api**: For gRPC communication (optional, for advanced querying)

In [ ]:
# Import necessary libraries
import tensorflow as tf
from tensorflow import keras
import numpy as np
import os
import json
import requests
import matplotlib.pyplot as plt

# Check TensorFlow version
print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

## Section 2: Creating and Training a Simple Model

We'll create a simple neural network for **MNIST digit classification**. This will be our example model that we'll deploy using TensorFlow Serving.

### About MNIST Dataset:
- **28×28 grayscale images** of handwritten digits (0-9)
- **60,000 training images** and **10,000 test images**
- Classic benchmark for image classification

In [ ]:
# Load and preprocess MNIST dataset
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# Normalize pixel values to [0, 1] range
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Reshape to add channel dimension (needed for some models)
X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Training labels shape: {y_train.shape}")

# Visualize some examples
plt.figure(figsize=(10, 2))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(X_train[i].squeeze(), cmap='gray')
    plt.title(str(y_train[i]))
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Create a simple Sequential model
model = keras.models.Sequential([
    # Flatten the 28x28 image to a 784-dimensional vector
    keras.layers.Flatten(input_shape=[28, 28, 1]),
    
    # Hidden layer with 300 neurons and ReLU activation
    keras.layers.Dense(300, activation='relu'),
    
    # Dropout for regularization (reduce overfitting)
    keras.layers.Dropout(0.2),
    
    # Hidden layer with 100 neurons
    keras.layers.Dense(100, activation='relu'),
    
    # Dropout layer
    keras.layers.Dropout(0.2),
    
    # Output layer: 10 classes (digits 0-9) with softmax
    keras.layers.Dense(10, activation='softmax')
])

# Display model architecture
model.summary()

In [ ]:
# Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # For integer labels
    metrics=['accuracy']
)

# Train the model
print("Training the model...")
history = model.fit(
    X_train, y_train,
    epochs=5,
    validation_split=0.1,
    batch_size=128,
    verbose=1
)

In [ ]:
# Evaluate the model on test set
print("\nEvaluating the model on test data...")
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test loss: {test_loss:.4f}")

# Make predictions on a few test samples
X_new = X_test[:3]
y_true = y_test[:3]

# Predict
y_pred_proba = model.predict(X_new)
y_pred = np.argmax(y_pred_proba, axis=1)

# Visualize predictions
plt.figure(figsize=(12, 4))
for i in range(3):
    plt.subplot(1, 3, i + 1)
    plt.imshow(X_new[i].squeeze(), cmap='gray')
    plt.title(f"True: {y_true[i]}, Pred: {y_pred[i]}\nConfidence: {y_pred_proba[i, y_pred[i]]:.2f}")
    plt.axis('off')
plt.tight_layout()
plt.show()

print(f"\nPredictions: {y_pred}")
print(f"True labels: {y_true}")

## Section 3: Exporting Models to SavedModel Format

### What is SavedModel?

**SavedModel** is TensorFlow's universal serialization format for saving and loading models. It includes:
- 📦 **Model Architecture**: The computation graph
- ⚖️ **Trained Weights**: All model parameters
- 🎯 **Signatures**: How to call the model (input/output specs)
- 📊 **Assets**: Additional files (vocabularies, etc.)

### SavedModel Directory Structure

When you export a model, TensorFlow creates this structure:
```
my_mnist_model/
└── 0001/                    ← Version number
    ├── assets/              ← Additional files
    ├── saved_model.pb       ← The model graph and metadata
    └── variables/           ← Model weights
        ├── variables.data-00000-of-00001
        └── variables.index
```

### Best Practice
Include preprocessing in your model so clients don't need to replicate preprocessing logic!